In [ ]:
import os
at_colab = 'COLAB_GPU' in os.environ

if at_colab:
    %pip install pygadm
    %pip install owslib
    %pip install distinctipy
    %pip install folium
    %pip install mapclassify
    %pip install selenium webdriver-manager
    %pip install wget

In [ ]:
import os
import wget

url = 'https://raw.githubusercontent.com/gromicho/tools/main/jg_folium.py'
filename = 'jg_folium.py'

# Ensure the file is removed before downloading
if os.path.exists(filename):
    os.remove(filename)

# Download the file (this will now always download the latest version)
wget.download(url, filename)

In [ ]:
import jg_folium as jg

In [ ]:
# Standard library imports
import xml.etree.ElementTree as ET
import requests
import io
from itertools import chain

# Third-party library imports
import geopandas as gpd
import distinctipy
import matplotlib.colors as mcolors
from owslib.wfs import WebFeatureService

In [ ]:
# Define the WFS GetCapabilities URL
wfs_url = 'https://service.pdok.nl/cbs/gebiedsindelingen/2025/wfs/v1_0'
get_capabilities_url = f'{wfs_url}?service=WFS&version=2.0.0&request=GetCapabilities'

# Download the GetCapabilities XML document
response = requests.get(get_capabilities_url)

if response.status_code == 200:
    # Parse the XML content
    root = ET.fromstring(response.content)

    # Extract all FeatureType names
    namespaces = {'wfs': 'http://www.opengis.net/wfs/2.0'}
    feature_types = root.findall('.//wfs:FeatureType', namespaces)
else:
    print(f'Failed to download GetCapabilities XML. HTTP Status Code: {response.status_code}')


In [ ]:
# Extract and filter FeatureType names using list comprehension
filtered_names = [
    feature.find('wfs:Name', namespaces).text
    for feature in feature_types
    if feature.find('wfs:Name', namespaces) is not None
]

In [ ]:
def from_pdok(
    layer_name,
    wfs_url = 'https://service.pdok.nl/cbs/gebiedsindelingen/2025/wfs/v1_0'
):
    # Connect to the WFS service
    wfs = WebFeatureService(url=wfs_url, version='2.0.0')

    # Fetch the data as GeoJSON
    params = {
        'service': 'WFS',
        'version': '2.0.0',
        'request': 'GetFeature',
        'typeName': layer_name,
        'outputFormat': 'application/json'
    }

    # Make the request
    response = requests.get(wfs_url, params=params)

    # Check if request was successful
    if response.status_code == 200:
        # Load response into a GeoDataFrame
        return gpd.read_file(io.StringIO(response.text))
    else:
        print(f'WFS request failed with status code {response.status_code}')
        print(f'Error message: {response.text}')
    return None

In [ ]:
cbs = { name : from_pdok(name) for name in filtered_names }

In [ ]:
[f'crs={df.crs.to_epsg()} @ {n}' for n,df in cbs.items() ]

In [ ]:
gdf_country = cbs['gebiedsindelingen:landsdeel_gegeneraliseerd'].dissolve().copy()
gdf_provinces = cbs['gebiedsindelingen:provincie_gegeneraliseerd'].copy()
gdf_cities = cbs['gebiedsindelingen:gemeente_gegeneraliseerd'].copy()

In [ ]:
gdf_country.explore()

In [ ]:
# Backup original geometry
gdf_provinces['geometry_original'] = gdf_provinces['geometry'].copy()

# Apply buffer
gdf_provinces['geometry'] = gdf_provinces.buffer(0.001)

# Perform spatial join
gdf_cities_with_province = gpd.sjoin(gdf_cities, gdf_provinces, how='left', predicate='within').drop(columns=['index_right'])

# Restore the original geometry
gdf_provinces['geometry'] = gdf_provinces['geometry_original']

# Drop the temporary backup column if not needed
gdf_provinces = gdf_provinces.drop(columns=['geometry_original'])

In [ ]:
gdf = gdf_cities_with_province.rename(
    columns={'statnaam_right':'Province','statnaam_left':'Municipality'}
).sort_values(['Province','Municipality'])

In [ ]:
province_to_city = gdf.groupby('Province')['Municipality'].apply(list).to_dict()

In [ ]:
assert set(gdf_cities.statnaam) == set(chain(*province_to_city.values()))

In [ ]:
seen = set()
duplicates = False
for lst in province_to_city.values():
    if (intersection := seen & set(lst)):  # Check for intersection
        print(sorted(intersection))
        duplicates = True
    seen.update(lst)  # Add elements to the global set
assert not duplicates

In [ ]:
city_names = sorted(set(gdf.Municipality))
city_colors = distinctipy.get_colors(len(city_names))

gdf['color_by_city'] = gdf['Municipality'].map(
    {
        name: mcolors.to_hex(color)
        for color, name in zip(city_colors, city_names)
    }
)

In [ ]:
map_all_cities = gdf.explore(
    color=gdf['color_by_city'],
    legend=False,
    tiles='OpenStreetMap',
    tooltip=['Municipality','Province'],
    zoom_on_bounds=True,
    style_kwds={"color": "black", "weight": 1}
)
jg.FoliumToPng( map_all_cities, 'map_all_cities_cbs' )
map_all_cities

In [ ]:
map_all_provinces = gdf.explore(
    column='Province',
    cmap='Paired',
    legend=True,
    categorical=True,
    tiles='OpenStreetMap',
    tooltip=['Municipality', 'Province'],
    zoom_on_bounds=True,
    style_kwds={"color": "black", "weight": 1}
)
jg.FoliumToPng( map_all_provinces, 'map_all_provinces_cbs' )
map_all_provinces

In [ ]:
map_capelle = gdf[gdf.Municipality.str.contains('Capelle')].explore(
    tooltip=False,
    style_kwds={'color': 'green', 'weight': 5, 'fill': False}
)
jg.FoliumToPng( map_capelle, 'map_capelle_cbs' )
map_capelle

In [ ]:
if at_colab:
    from google.colab import files
    files.download('map_all_cities_cbs.png')
    files.download('map_all_provinces_cbs.png')
    files.download('map_capelle_cbs.png')
else:
    import pyperclip
    pyperclip.copy('\r\n'.join(sorted(gdf.Municipality.values)))

In [ ]:
gdf.shape

In [ ]:
gdf.to_parquet('cbs.parquet')
gdf_country.to_parquet('cbs_nl.parquet')